In [ ]:
# Cell 1: Imports
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, ConcatDataset

import cv2
import numpy as np
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Cấu hình Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {device}")

In [ ]:
# Cell 2: Hyperparameters
LEARNING_RATE = 1e-4
BATCH_SIZE = 16
IMAGE_SIZE = 416 # YOLOv3 chuẩn thường dùng 416x416
NUM_CLASSES = 5 # Ví dụ: Xe hơi, Xe buýt, Xe tải, Xe máy, Xe đạp
EPOCHS = 20

# Đường dẫn dữ liệu (Tùy chỉnh nếu bạn đang mount data trên Kaggle hoặc ổ cứng cục bộ)
DATA_DIR = "/kaggle/input/datasets/marquis03/bdd100k/"
IMG_DIR = os.path.join(DATA_DIR, "train/images")
LABEL_FILE = os.path.join(DATA_DIR, "train/annotations/bdd100k_labels_images_train.json")

# Anchor boxes của YOLOv3 (Chuẩn hóa)
ANCHORS = [
    [(0.28, 0.22), (0.38, 0.48), (0.9, 0.78)],
    [(0.07, 0.15), (0.15, 0.11), (0.14, 0.29)],
    [(0.02, 0.03), (0.04, 0.07), (0.08, 0.06)],
]

In [ ]:
def iou_width_height(boxes1, boxes2):
    """
    Tính IoU dựa trên width và height.
    boxes1: tensor của các bounding box thật (w, h)
    boxes2: tensor của các anchor boxes (w, h)
    """
    intersection = torch.min(boxes1[..., 0], boxes2[..., 0]) * torch.min(boxes1[..., 1], boxes2[..., 1])
    union = (boxes1[..., 0] * boxes1[..., 1] + boxes2[..., 0] * boxes2[..., 1] - intersection)
    return intersection / union
    

In [ ]:
# Cell 3.5: Cài đặt và import thư viện (Chạy lệnh pip nếu chưa cài)
# !pip install albumentations

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# Cấu hình chuẩn của YOLOv3 (Darknet-53 + FPN)
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 1],
    (128, 3, 2),
    ["B", 2],
    (256, 3, 2),
    ["B", 8],
    # Nhánh đầu tiên đi ra từ đây (Dành cho scale 52x52)
    (512, 3, 2),
    ["B", 8],
    # Nhánh thứ hai đi ra từ đây (Dành cho scale 26x26)
    (1024, 3, 2),
    ["B", 4],
    # Đến đây là hết Darknet-53
    (512, 1, 1),
    (1024, 3, 1),
    "S", # Dự đoán Scale 1 (13x13) - Vật thể lớn
    (256, 1, 1),
    "U", # Upsampling
    (256, 1, 1),
    (512, 3, 1),
    "S", # Dự đoán Scale 2 (26x26) - Vật thể vừa
    (128, 1, 1),
    "U", # Upsampling
    (128, 1, 1),
    (256, 3, 1),
    "S", # Dự đoán Scale 3 (52x52) - Vật thể nhỏ
]
IMAGE_SIZE = 416 # Kích thước đầu vào chuẩn của YOLOv3

# 1. Transform cho tập Train (Data Augmentation)
train_transforms = A.Compose(
    [
        # Resize ảnh về kích thước chuẩn để đưa vào mạng CNN
        A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),

        # Lật ngang ảnh ngẫu nhiên (xác suất 50%)
        # Rất hữu ích cho BDD100K vì các phương tiện nhìn từ trái hay phải đều hợp lệ
        A.HorizontalFlip(p=0.5),

        # Thay đổi ngẫu nhiên độ sáng, độ tương phản và màu sắc để model chịu lỗi tốt hơn
        # với các điều kiện thời tiết/ánh sáng khác nhau trong BDD100K
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),

        # Thêm hiệu ứng mờ nhòe ngẫu nhiên (giả lập camera rung hoặc xe di chuyển nhanh)
        A.OneOf([
            A.MotionBlur(p=0.2),
            A.MedianBlur(blur_limit=3, p=0.1),
            A.Blur(blur_limit=3, p=0.1),
        ], p=0.2),

        # Chuẩn hóa giá trị pixel (Normalize) về khoảng [-1, 1] hoặc tương tự
        # Sử dụng mean và std chuẩn của tập ImageNet (do Darknet-53 thường pre-train trên ImageNet)
        A.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225], 
            max_pixel_value=255.0
        ),

        # Chuyển đổi numpy array (H, W, C) sang PyTorch Tensor (C, H, W)
        ToTensorV2(),
    ],
    
    # [QUAN TRỌNG] Cấu hình xử lý Bounding Box
    bbox_params=A.BboxParams(
        format="yolo", # Bắt buộc phải là "yolo" vì dataset của ta trả về [x_center, y_center, w, h] chuẩn hóa
        min_visibility=0.4, # Nếu ảnh bị cắt (crop) làm mất quá 60% diện tích box, box đó sẽ bị loại bỏ
        label_fields=["class_labels"], # Khai báo tên biến chứa nhãn (class) đi kèm với box
    ),
)

# 2. Transform cho tập Validation / Test
# Tập này KHÔNG được phép thêm nhiễu hay lật ảnh, chỉ cần đưa về đúng định dạng model cần
val_transforms = A.Compose(
    [
        A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
        A.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225], 
            max_pixel_value=255.0
        ),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        min_visibility=0.4,
        label_fields=["class_labels"],
    ),
)

In [ ]:
class COCOVehicleDataset(Dataset):
    def __init__(self, json_file, img_dir, anchors, image_size=416, S=[13, 26, 52], transform=None):
        self.img_dir = img_dir
        self.image_size = image_size
        self.transform = transform
        self.S = S
        self.anchors = torch.tensor(anchors[0] + anchors[1] + anchors[2])
        self.num_anchors = self.anchors.shape[0]
        self.num_anchors_per_scale = self.num_anchors // 3
        
        # MAPPING QUAN TRỌNG: 
        # Cần map các class của COCO về đúng ID mà YOLOv3 đang dùng.
        # Ở bài trước, ta định nghĩa classes = ['car', 'bus', 'truck', 'motorcycle', 'bicycle']
        # Do đó: motorcycle = 3, bicycle = 4
        self.target_classes = {'motorcycle': 3, 'bicycle': 4}
        
        self.data = self._parse_coco_json(json_file)

    def _parse_coco_json(self, json_file):
        print(f"Đang đọc file JSON của COCO: {json_file}...")
        with open(json_file, 'r') as f:
            coco_data = json.load(f)

        # 1. Lấy thông tin Categories và tạo mapping từ Category ID sang Tên class
        categories = coco_data['categories']
        cat_id_to_name = {cat['id']: cat['name'] for cat in categories}
        
        # Lọc ra ID của các class ta quan tâm trong bộ COCO
        target_cat_ids = [cat_id for cat_id, name in cat_id_to_name.items() if name in self.target_classes.keys()]
        
        print(f"Tìm thấy Category IDs cần thiết trong COCO: {target_cat_ids}")

        # 2. Xây dựng dictionary gom nhóm Annotations theo Image ID
        # Khác với BDD, COCO lưu annotation riêng rẽ. Ta cần gom chúng lại theo từng ảnh.
        image_annotations = {}
        for ann in coco_data['annotations']:
            if ann['category_id'] in target_cat_ids:
                img_id = ann['image_id']
                if img_id not in image_annotations:
                    image_annotations[img_id] = []
                image_annotations[img_id].append(ann)

        # 3. Kết hợp với thông tin ảnh và chuyển đổi định dạng Bounding Box
        parsed_data = []
        for img_info in coco_data['images']:
            img_id = img_info['id']
            
            # Chỉ lấy những ảnh có chứa annotation của xe đạp/xe máy
            if img_id in image_annotations:
                img_name = img_info['file_name']
                orig_w = float(img_info['width'])
                orig_h = float(img_info['height'])
                
                bboxes = []
                labels = []
                
                for ann in image_annotations[img_id]:
                    # Bounding Box của COCO có định dạng [x_min, y_min, width, height] (tọa độ pixel tuyệt đối)
                    x_min, y_min, bbox_w, bbox_h = ann['bbox']
                    
                    # Chuyển đổi sang định dạng YOLO chuẩn hóa (0-1): [x_center, y_center, w, h]
                    x_center = (x_min + bbox_w / 2.0) / orig_w
                    y_center = (y_min + bbox_h / 2.0) / orig_h
                    width = bbox_w / orig_w
                    height = bbox_h / orig_h
                    
                    # Lấy class name từ COCO, sau đó chuyển sang ID của YOLO model
                    cat_name = cat_id_to_name[ann['category_id']]
                    yolo_class_id = self.target_classes[cat_name]
                    
                    bboxes.append([x_center, y_center, width, height])
                    labels.append(yolo_class_id)
                
                parsed_data.append({
                    'img_name': img_name,
                    'bboxes': bboxes,
                    'labels': labels
                })
                
        print(f"Hoàn thành! Tìm thấy {len(parsed_data)} ảnh COCO chứa xe đạp/xe máy.")
        return parsed_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        # --- Hàm này giống hệt như trong BDD100KDataset ---
        img_info = self.data[index]
        img_path = os.path.join(self.img_dir, img_info['img_name'])
        
        image = cv2.imread(img_path)
        
        if image is None:
            print(f"Cảnh báo: Không thể đọc ảnh COCO {img_path}. Bỏ qua.")
            import random
            return self.__getitem__(random.randint(0, len(self.data) - 1))
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        bboxes = np.array(img_info['bboxes'])
        labels = np.array(img_info['labels'])
        
        if self.transform:
            augmentations = self.transform(image=image, bboxes=bboxes, class_labels=labels)
            image = augmentations["image"]
            bboxes = augmentations["bboxes"]
            labels = augmentations["class_labels"]

        # Xây dựng ma trận Target (Logic giống hệt BDD100KDataset)
        targets = [torch.zeros((self.num_anchors_per_scale, S, S, 6)) for S in self.S]
        
        for box, class_label in zip(bboxes, labels):
            x, y, w, h = box
            iou_anchors = iou_width_height(torch.tensor([w, h]), self.anchors)
            anchor_indices = iou_anchors.argsort(descending=True, dim=0)
            has_anchor = [False, False, False]
            
            for anchor_idx in anchor_indices:
                scale_idx = anchor_idx // self.num_anchors_per_scale
                anchor_on_scale = anchor_idx % self.num_anchors_per_scale
                S = self.S[scale_idx]
                i, j = int(S * y), int(S * x)
                i = min(i, S - 1)
                j = min(j, S - 1)
                
                anchor_taken = targets[scale_idx][anchor_on_scale, i, j, 0]
                
                if not anchor_taken and not has_anchor[scale_idx]:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = 1
                    x_cell, y_cell = S * x - j, S * y - i
                    width_cell, height_cell = (w * S, h * S)
                    box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])
                    targets[scale_idx][anchor_on_scale, i, j, 1:5] = box_coordinates
                    targets[scale_idx][anchor_on_scale, i, j, 5] = int(class_label)
                    has_anchor[scale_idx] = True
                elif not anchor_taken and iou_anchors[anchor_idx] > 0.5:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = -1

        return image, tuple(targets)

In [ ]:
class BDD100KDataset(Dataset):
    def __init__(self, json_file, img_dir, anchors, image_size=416, S=[13, 26, 52], transform=None):
        self.img_dir = img_dir
        self.image_size = image_size
        self.transform = transform
        self.S = S # Grid sizes cho 3 scale
        
        # YOLOv3 có 3 scale, mỗi scale có 3 anchors
        self.anchors = torch.tensor(anchors[0] + anchors[1] + anchors[2]) 
        self.num_anchors = self.anchors.shape[0] # Tổng 9 anchors
        self.num_anchors_per_scale = self.num_anchors // 3
        
        # Định nghĩa các class cần nhận dạng
        self.classes = ['car', 'bus', 'truck', 'motorcycle', 'bicycle']
        self.class_to_id = {cls: i for i, cls in enumerate(self.classes)}
        
        # Parse JSON
        self.data = self._parse_json(json_file)

    def _parse_json(self, json_file):
        print(f"Đang đọc file JSON: {json_file}...")
        with open(json_file, 'r') as f:
            bdd_labels = json.load(f)
            
        parsed_data = []
        for item in bdd_labels:
            img_name = item['name']
            bboxes = []
            labels = []
            
            # Bỏ qua những ảnh không có nhãn
            if 'labels' not in item:
                continue
                
            for label in item['labels']:
                category = label['category']
                
                # Chỉ lấy các object thuộc nhóm phương tiện
                if category in self.classes and 'box2d' in label:
                    box2d = label['box2d']
                    x1, y1, x2, y2 = box2d['x1'], box2d['y1'], box2d['x2'], box2d['y2']
                    
                    # YOLO format: x_center, y_center, width, height (Chuẩn hóa 0-1)
                    # BDD100K ảnh gốc thường là 1280x720
                    orig_w, orig_h = 1280.0, 720.0 
                    
                    x_center = ((x1 + x2) / 2) / orig_w
                    y_center = ((y1 + y2) / 2) / orig_h
                    width = (x2 - x1) / orig_w
                    height = (y2 - y1) / orig_h
                    
                    bboxes.append([x_center, y_center, width, height])
                    labels.append(self.class_to_id[category])
            
            # Chỉ lưu ảnh có chứa phương tiện để tiết kiệm thời gian train
            if len(bboxes) > 0:
                parsed_data.append({
                    'img_name': img_name,
                    'bboxes': bboxes,
                    'labels': labels
                })
        print(f"Hoàn thành! Tìm thấy {len(parsed_data)} ảnh chứa phương tiện.")
        return parsed_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_info = self.data[index]
        img_path = os.path.join(self.img_dir, img_info['img_name'])
        
        # Đọc ảnh bằng OpenCV và chuyển sang RGB
        image = cv2.imread(img_path)
        
        if image is None:
            print(f"Cảnh báo: Không thể đọc ảnh {img_path}. Bỏ qua và lấy ảnh khác.")
            # Nếu ảnh lỗi, tự động lấy random một ảnh khác trong dataset để thay thế
            # Điều này giúp batch size luôn được giữ nguyên và model không bị sập
            import random
            random_index = random.randint(0, len(self.data) - 1)
            return self.__getitem__(random_index)
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        bboxes = np.array(img_info['bboxes'])
        labels = np.array(img_info['labels'])
        
        # Nếu có Transform (như Albumentations), áp dụng tại đây
        if self.transform:
            augmentations = self.transform(image=image, bboxes=bboxes, class_labels=labels)
            image = augmentations["image"]
            bboxes = augmentations["bboxes"]
            labels = augmentations["class_labels"]

        # Xây dựng ma trận Target cho 3 Scale
        # Shape của target mỗi scale: (num_anchors_per_scale, grid_size, grid_size, 6)
        # 6 elements: [obj_prob, x, y, w, h, class_label]
        targets = [torch.zeros((self.num_anchors_per_scale, S, S, 6)) for S in self.S]
        
        for box, class_label in zip(bboxes, labels):
            x, y, w, h = box
            
            # Tính IoU của bbox với 9 anchors
            iou_anchors = iou_width_height(torch.tensor([w, h]), self.anchors)
            # Sắp xếp anchor phù hợp nhất từ cao xuống thấp
            anchor_indices = iou_anchors.argsort(descending=True, dim=0)
            
            has_anchor = [False, False, False]  # Đảm bảo mỗi scale chỉ nhận 1 anchor tốt nhất
            
            for anchor_idx in anchor_indices:
                # Tìm xem anchor này thuộc scale nào (0, 1 hay 2)
                scale_idx = anchor_idx // self.num_anchors_per_scale
                anchor_on_scale = anchor_idx % self.num_anchors_per_scale
                
                S = self.S[scale_idx]
                
                # Tìm tọa độ ô Grid cell (i, j) mà tâm của bounding box rơi vào
                i, j = int(S * y), int(S * x)
                
                # Tránh lỗi index out of bounds khi x, y = 1.0
                i = min(i, S - 1)
                j = min(j, S - 1)
                
                # Kiểm tra xem ô lưới này cho anchor này đã có object chưa
                anchor_taken = targets[scale_idx][anchor_on_scale, i, j, 0]
                
                if not anchor_taken and not has_anchor[scale_idx]:
                    # Đánh dấu đã gán
                    targets[scale_idx][anchor_on_scale, i, j, 0] = 1
                    
                    # Tính toán tọa độ x, y tương đối so với ô lưới
                    x_cell, y_cell = S * x - j, S * y - i
                    
                    # Lưu kích thước w, h (tương đối so với toàn bộ ảnh, loss function sẽ xử lý log)
                    width_cell, height_cell = (w * S, h * S)
                    
                    # Gán giá trị vào tensor
                    box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])
                    targets[scale_idx][anchor_on_scale, i, j, 1:5] = box_coordinates
                    targets[scale_idx][anchor_on_scale, i, j, 5] = int(class_label)
                    
                    has_anchor[scale_idx] = True
                
                # Nếu một anchor khác cũng dự đoán tốt nhưng ô lưới đã bị chiếm
                # Ta set obj_prob = -1 để bỏ qua (ignore) trong quá trình tính Loss (tránh phạt sai)
                elif not anchor_taken and iou_anchors[anchor_idx] > 0.5:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = -1

        return image, tuple(targets)

In [ ]:
# Ví dụ khởi tạo (Bạn sẽ đặt đoạn này ở Cell tiếp theo sau khi định nghĩa xong class Dataset)
train_dataset = BDD100KDataset(
    json_file=LABEL_FILE, # Đường dẫn file JSON train
    img_dir=IMG_DIR,      # Đường dẫn thư mục ảnh train
    anchors=ANCHORS,
    image_size=IMAGE_SIZE,
    transform=train_transforms # Truyền bộ transform train vào đây
)

# Thử in ra một sample để kiểm tra
# img, targets = train_dataset[0]
# print("Image shape:", img.shape) # Output mong đợi: torch.Size([3, 416, 416])

In [ ]:
# Cell 4: Model Blocks
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        # Tính toán padding để giữ nguyên kích thước nếu stride=1
        # padding = (kernel_size - 1) // 2 (với điều kiện ảnh vuông, padding đều)
        self.conv = nn.Conv2d(in_channels, out_channels, bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels) if bn_act else nn.Identity()
        self.leaky = nn.LeakyReLU(0.1) if bn_act else nn.Identity()

    def forward(self, x):
        return self.leaky(self.bn(self.conv(x)))


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_repeats):
            self.layers.append(
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            )
        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x) # Skip connection: Cộng trực tiếp input vào output
            else:
                x = layer(x)
        return x


class ScalePrediction(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, in_channels * 2, kernel_size=3, padding=1),
            # Lớp cuối không dùng BatchNorm và Activation (bn_act=False)
            # Output channels = 3 anchors * (5 bounding box info + num_classes)
            CNNBlock(
                in_channels * 2, (num_classes + 5) * 3, bn_act=False, kernel_size=1
            ),
        )
        self.num_classes = num_classes

    def forward(self, x):
        # x shape: (batch_size, 3 * (num_classes + 5), grid_size, grid_size)
        # Cần reshape lại để dễ tính Loss: (batch_size, 3, grid_size, grid_size, num_classes + 5)
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )

In [ ]:
class YOLOv3(nn.Module):
    def __init__(self, in_channels=3, num_classes=5): # Nhớ setup num_classes=5 cho BDD100K xe cộ
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []  # Chứa kết quả dự đoán của 3 scale
        route_connections = [] # Chứa các feature map để nối (skip connection từ Darknet)

        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            # Lưu lại feature map ở 2 vị trí quan trọng của Darknet-53 để nối nhánh (FPN)
            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)
            
            # Khớp nhánh Upsampling với feature map đã lưu
            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1) # Nối theo channel
                route_connections.pop()

        return outputs # Trả về mảng chứa 3 tensor: [Scale 13x13, Scale 26x26, Scale 52x52]

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels

        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0,
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(in_channels, num_repeats=num_repeats))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels // 2, num_classes=self.num_classes),
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2))
                    in_channels = in_channels * 3 # Do nối (cat) x với nhánh route_connection có số channel gấp đôi

        return layers

In [ ]:
# Cell test model
if __name__ == "__main__":
    num_classes = 5
    IMAGE_SIZE = 416
    
    # Tạo một tensor ảnh ảo kích thước 416x416, batch size = 2
    x = torch.randn((2, 3, IMAGE_SIZE, IMAGE_SIZE))
    model = YOLOv3(num_classes=num_classes)
    
    # Chạy forward pass
    out = model(x)
    
    print(f"Kiểm tra kích thước Output:")
    print(f"Scale 1 (13x13): {out[0].shape}") 
    # Output mong đợi: torch.Size([2, 3, 13, 13, 10]) -> 10 = 5 bounding box info + 5 classes
    print(f"Scale 2 (26x26): {out[1].shape}")
    print(f"Scale 3 (52x52): {out[2].shape}")

Các lưu ý cực kỳ quan trọng về hàm Loss này:Chống Nổ Gradient (Exploding Gradient): Hàm torch.exp() cho Width và Height rất dễ sinh ra các giá trị vô cực (Infinity) ở những Epoch đầu tiên khi model đang dự đoán ngẫu nhiên. Dòng torch.clamp(..., max=1e3) là một thủ thuật thực chiến nhỏ (trick) để bảo vệ model không bị sập (trả về NaN loss).BCEWithLogitsLoss: Khi tính Object và No-Object Loss, ta truyền thẳng predictions[..., 0:1] (logit thô chưa qua hàm kích hoạt) vào hàm self.bce. Tuyệt đối không dùng self.sigmoid ở bước này vì PyTorch đã tự động tối ưu quá trình đó bên trong BCEWithLogitsLoss cho tốc độ và độ chính xác dấu phẩy động tốt hơn.Khớp kích thước Anchors: Biến anchors truyền vào hàm forward phải là anchors đã được chia tỷ lệ (scaled) theo grid size (ví dụ $13 \times 13$) chứ không phải anchors gốc của ảnh $416 \times 416$. Chúng ta sẽ xử lý việc chia tỷ lệ này trong Training Loop.

In [ ]:
# Cell 5: Loss Function
class YoloLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Hàm Loss cho tọa độ và kích thước (x, y, w, h)
        self.mse = nn.MSELoss()
        
        # Hàm Loss cho Objectness (Có vật thể hay không) 
        # Dùng BCEWithLogitsLoss thay vì BCE thường vì nó đã tích hợp sẵn Sigmoid, giúp tính toán ổn định hơn (tránh gradient nổ)
        self.bce = nn.BCEWithLogitsLoss()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        class_weights = torch.tensor([1.0, 5.0, 5.0, 10.0, 10.0]).to(device)
        
        # Hàm Loss cho Phân loại (Class)
        self.cross_entropy = nn.CrossEntropyLoss(weight=class_weights)
        
        self.sigmoid = nn.Sigmoid()

        # Các hằng số trọng số (Lambda) để cân bằng các thành phần Loss
        # Vì số lượng ô không có vật thể (background) luôn áp đảo, ta cần phạt No-Object nhẹ hơn 
        # và ưu tiên phạt/thưởng Box tọa độ nặng hơn.
        self.lambda_class = 1
        self.lambda_noobj = 10
        self.lambda_obj = 1
        self.lambda_box = 10

    def forward(self, predictions, target, anchors):
        """
        predictions shape: (BATCH_SIZE, 3, S, S, 10) -> 10 = 1(obj) + 4(box) + 5(classes)
        target shape: (BATCH_SIZE, 3, S, S, 6) -> 6 = 1(obj) + 4(box) + 1(class_label)
        anchors shape: (3, 2) -> 3 anchors cho scale hiện tại, mỗi anchor có (w, h)
        """
        # 1. Xác định các Mask (Mặt nạ) để lọc dữ liệu
        # obj: Ô lưới CÓ chứa vật thể (target[..., 0] == 1)
        # noobj: Ô lưới KHÔNG chứa vật thể (target[..., 0] == 0)
        # Lưu ý: Các ô bị gán -1 (ignore) sẽ tự động bị loại trừ khỏi cả 2 mask này
        obj = target[..., 0] == 1
        noobj = target[..., 0] == 0

        # ======================== #
        #   A. NO-OBJECT LOSS      #
        # ======================== #
        # Ép xác suất dự đoán của các ô 'noobj' về 0
        no_object_loss = self.bce(
            (predictions[..., 0:1][noobj]), (target[..., 0:1][noobj])
        )

        # ======================== #
        #   B. OBJECT LOSS         #
        # ======================== #
        # Ép xác suất dự đoán của các ô 'obj' về 1, nhân với IoU để model học cách dự đoán độ tin cậy thực tế
        # Việc lấy IoU ở đây là nâng cao (tùy chọn), nhưng nó giúp YOLOv3 chính xác hơn.
        # Để đơn giản hóa ở phiên bản từ đầu, ta ép thẳng về 1 (target[..., 0:1][obj])
        object_loss = self.bce(
            (predictions[..., 0:1][obj]), (target[..., 0:1][obj])
        )

        # ======================== #
        #   C. BOX COORDINATE LOSS #
        # ======================== #
        # 1. Tính X, Y (Áp dụng Sigmoid cho x, y)
        xy_preds = self.sigmoid(predictions[..., 1:3][obj])
        
        # 2. Tính W, H (Áp dụng Exp và nhân với anchors để lấy width, height)
        # Reshape anchors và tự động broadcast trước khi áp dụng mask [obj]
        anchors = anchors.reshape(1, 3, 1, 1, 2)
        wh_preds = (torch.exp(predictions[..., 3:5]) * anchors)[obj]
        
        # Kẹp giá trị (clamp) để chống nổ Gradient (NaN) do hàm Exp
        wh_preds = torch.clamp(wh_preds, max=1e3)

        # 3. Gộp X,Y và W,H lại với nhau để tạo thành box dự đoán hoàn chỉnh
        # dim=-1 nghĩa là nối theo chiều cuối cùng (từ 2 tensor có shape là 2 -> thành tensor shape 4)
        box_preds = torch.cat([xy_preds, wh_preds], dim=-1)

        # 4. Lấy target thực tế và tính MSE Loss
        target_box = target[..., 1:5][obj]
        box_loss = self.mse(box_preds, target_box)
        # ======================== #
        #   D. CLASS LOSS          #
        # ======================== #
        # Dự đoán phân loại: từ index 5 đến cuối
        # Target phân loại: nằm ở index 5. Cần chuyển sang kiểu Long (số nguyên) cho hàm CrossEntropy
        class_loss = self.cross_entropy(
            (predictions[..., 5:][obj]), (target[..., 5][obj].long())
        )

        # ======================== #
        #   TỔNG HỢP LOSS          #
        # ======================== #
        total_loss = (
            self.lambda_box * box_loss
            + self.lambda_obj * object_loss
            + self.lambda_noobj * no_object_loss
            + self.lambda_class * class_loss
        )

        return total_loss

In [ ]:
# Cell 6: Utilities

def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    Tính toán chỉ số IoU
    boxes_preds: tensor dự đoán (BATCH_SIZE, 4)
    boxes_labels: tensor nhãn thực tế (BATCH_SIZE, 4)
    box_format: "midpoint" (x, y, w, h) hoặc "corners" (x1, y1, x2, y2)
    """

    if box_format == "midpoint":
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2
        
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    if box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]
        
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    # Tìm tọa độ của hình chữ nhật giao nhau
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # .clamp(0) đảm bảo nếu không giao nhau thì diện tích = 0 (tránh số âm)
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    # Tính diện tích của cả 2 box
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    # Công thức: Giao / (Diện tích 1 + Diện tích 2 - Giao)
    # Cộng thêm 1e-6 để tránh lỗi chia cho 0
    return intersection / (box1_area + box2_area - intersection + 1e-6)

def non_max_suppression(bboxes, iou_threshold, prob_threshold, box_format="corners"):
    """
    Thực hiện Non-Max Suppression cho list các bounding boxes.
    bboxes: danh sách các box với định dạng [class_pred, prob_score, x1, y1, x2, y2]
    iou_threshold: Ngưỡng IoU để quyết định 2 box có đang chỉ vào cùng 1 vật thể không
    prob_threshold: Ngưỡng loại bỏ các box dự đoán với độ tự tin thấp
    """
    # 1. Lọc bỏ các dự đoán quá kém (Dưới prob_threshold)
    bboxes = [box for box in bboxes if box[1] > prob_threshold]
    
    # 2. Sắp xếp danh sách box theo độ tự tin (prob_score) giảm dần
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)
    bboxes = bboxes[:100]
    
    bboxes_after_nms = []

    while bboxes:
        # Lấy box có độ tự tin cao nhất hiện tại (nằm ở đầu list do đã sort)
        chosen_box = bboxes.pop(0)

        # Lọc lại bboxes: Chỉ giữ lại những box KHÔNG chồng lấn quá nhiều với chosen_box
        # HOẶC giữ lại nếu chúng dự đoán các Class khác nhau (ví dụ 1 ô vừa dự đoán xe máy, vừa dự đoán người)
        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0] # Khác class thì giữ lại
            or intersection_over_union(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:]),
                box_format=box_format,
            )
            < iou_threshold # IoU nhỏ hơn ngưỡng (không chồng lấn nhiều) thì giữ lại
        ]

        # Đưa box tốt nhất này vào danh sách kết quả
        bboxes_after_nms.append(chosen_box)

    return bboxes_after_nms

def cells_to_bboxes(predictions, anchors, S, is_preds=True):
    """
    Chuyển đổi tensor đầu ra của mạng từ tọa độ tương đối theo ô lưới (Grid) 
    thành tọa độ tương đối so với toàn bộ kích thước ảnh (0-1).
    """
    BATCH_SIZE = predictions.shape[0]
    num_anchors = len(anchors)
    
    # Lấy các giá trị (Tương tự như trong YoloLoss)
    box_predictions = predictions[..., 1:5]
    if is_preds:
        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        
        # X, Y qua Sigmoid
        box_predictions[..., 0:2] = torch.sigmoid(box_predictions[..., 0:2])
        # W, H qua Exp nhân với anchors
        box_predictions[..., 2:] = torch.exp(box_predictions[..., 2:]) * anchors
        
        # Objectness score và Class probabilities
        scores = torch.sigmoid(predictions[..., 0:1])
        best_class = torch.argmax(predictions[..., 5:], dim=-1).unsqueeze(-1)
    else:
        # Nếu là Target (Nhãn thật) thì không cần sigmoid/exp
        scores = predictions[..., 0:1]
        best_class = predictions[..., 5:6]

    # Tính toán tọa độ x, y so với toàn bộ ảnh thay vì từng cell
    cell_indices = (
        torch.arange(S)
        .repeat(predictions.shape[0], 3, S, 1)
        .unsqueeze(-1)
        .to(predictions.device)
    )
    
    # Cộng thêm index của ô lưới và chia cho kích thước lưới S
    x = 1 / S * (box_predictions[..., 0:1] + cell_indices)
    y = 1 / S * (box_predictions[..., 1:2] + cell_indices.permute(0, 1, 3, 2, 4))
    
    # Kích thước W, H so với toàn bộ ảnh
    w_h = 1 / S * box_predictions[..., 2:4]

    # Gom lại thành dạng: [class, score, x, y, w, h]
    converted_bboxes = torch.cat((best_class, scores, x, y, w_h), dim=-1).reshape(BATCH_SIZE, num_anchors * S * S, 6)
    
    return converted_bboxes.tolist()

def plot_image(image, boxes):
    pass

In [ ]:
def train_fn(train_loader, model, optimizer, loss_fn, scaler, scaled_anchors):
    """
    Hàm huấn luyện cho 1 Epoch.
    """
    # Đưa một thanh tiến trình (progress bar) vào để dễ theo dõi
    loop = tqdm(train_loader, leave=True)
    losses = []

    for batch_idx, (x, y) in enumerate(loop):
        x = x.to(device)
        
        # y chứa 3 tensor target cho 3 scale khác nhau
        y0, y1, y2 = (
            y[0].to(device),
            y[1].to(device),
            y[2].to(device),
        )

        # Sử dụng Mixed Precision để tối ưu bộ nhớ và tốc độ
        with torch.cuda.amp.autocast():
            out = model(x)
            
            # Tính loss cho từng scale
            loss = (
                loss_fn(out[0], y0, scaled_anchors[0])
                + loss_fn(out[1], y1, scaled_anchors[1])
                + loss_fn(out[2], y2, scaled_anchors[2])
            )

        losses.append(loss.item())

        # Lan truyền ngược (Backpropagation) và cập nhật trọng số
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Cập nhật thanh tiến trình hiển thị Loss trung bình hiện tại
        mean_loss = sum(losses) / len(losses)
        loop.set_postfix(loss=mean_loss)

In [ ]:
import torch

def save_checkpoint(model, optimizer, filename="yolov3_checkpoint.pth"):
    """
    Hàm lưu lại toàn bộ trạng thái của mô hình và bộ tối ưu hóa.
    """
    print(f"=> Đang tiến hành lưu checkpoint vào tệp: {filename}...")
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    torch.save(checkpoint, filename)
    print("=> Đã lưu thành công!")

def load_checkpoint(checkpoint_file, model, optimizer=None, lr=None, device="cuda"):
    """
    Hàm tải trọng số vào mô hình. 
    Có thể dùng để train tiếp (cần truyền optimizer) hoặc chỉ để suy luận (chỉ cần model).
    """
    print(f"=> Đang tải checkpoint từ tệp: {checkpoint_file}...")
    checkpoint = torch.load(checkpoint_file, map_location=device)
    
    # Load trọng số vào model
    model.load_state_dict(checkpoint["state_dict"])
    
    # Nếu đang trong quá trình train tiếp, load thêm optimizer
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])
        
        # Cập nhật lại Learning Rate nếu bạn muốn đổi LR khi train tiếp
        if lr is not None:
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
                
    print("=> Tải checkpoint thành công!")
    saved_epoch = checkpoint.get("epoch", 0)
    
    return saved_epoch

In [ ]:
# Cell 9: Main Execution Loop
from tqdm import tqdm
import torch.optim as optim

 # 1. Khởi tạo Model và cấu hình cơ bản
model = YOLOv3(num_classes=NUM_CLASSES).to(device)
optimizer = optim.Adam(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
)
loss_fn = YoloLoss()
scaler = torch.cuda.amp.GradScaler()

# 2. Khởi tạo Dataloader cho tập Train
bdd_dataset = BDD100KDataset(
    json_file=LABEL_FILE, # Đường dẫn file JSON BDD
    img_dir=IMG_DIR,      # Đường dẫn ảnh BDD
    transform=train_transforms,
    S=[13, 26, 52],
    anchors=ANCHORS,
)

COCO_LABEL_FILE = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_train2017.json"
COCO_IMG_DIR = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"

coco_dataset = COCOVehicleDataset(
    json_file=COCO_LABEL_FILE,
    img_dir=COCO_IMG_DIR,
    transform=train_transforms, # Dùng chung bộ transform (Augmentation) với BDD
    S=[13, 26, 52],
    anchors=ANCHORS,
)

combined_dataset = ConcatDataset([bdd_dataset, coco_dataset])
print(f"Tổng số ảnh sau khi gộp: {len(combined_dataset)} (BDD: {len(bdd_dataset)} + COCO: {len(coco_dataset)})")

train_loader = DataLoader(
    dataset=combined_dataset,
    batch_size=BATCH_SIZE,
    num_workers=4, # Tăng số luồng đọc dữ liệu (tùy thuộc vào CPU của bạn/Kaggle)
    pin_memory=True,
    shuffle=True,
    drop_last=False,
)

# 3. Scale Anchors (Chuẩn hóa Anchors theo Grid Size)
# Đây là bước cực kỳ quan trọng. Anchor gốc nằm trong khoảng 0-1, 
# nhưng hàm Loss cần anchor tính theo ô lưới (ví dụ: nhân với 13, 26, 52)
S = [13, 26, 52]
scaled_anchors = (
    torch.tensor(ANCHORS)
    * torch.tensor(S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
).to(device)



In [ ]:
start_epoch = 0
checkpoint_file = "/kaggle/input/models/editcadic/bd100k-yolo-model/pytorch/default/4/yolov3_bdd100k (6).pth" # Đổi tên cho chuyên nghiệp
# CHỐT CHẶN AN TOÀN: Chỉ load nếu file có thật
if os.path.exists(checkpoint_file):
    print("--- Khôi phục quá trình huấn luyện từ đêm qua ---")
    # Giả định hàm load_checkpoint của bạn trả về epoch hiện tại
    start_epoch = load_checkpoint(checkpoint_file, model, optimizer)
    start_epoch += 1 # Bắt đầu từ epoch tiếp theo
else:
    print("--- Bắt đầu huấn luyện MỚI TỪ ĐẦU ---")
    
# Chúng ta dùng hàm load_checkpoint thay vì load thủ công để đảm bảo nạp đủ cả optimizer
load_checkpoint(checkpoint_file, model, optimizer=optimizer, device=device)

# 4. Bắt đầu vòng lặp Epochs
print(f"Bắt đầu huấn luyện với {EPOCHS} Epochs...")
for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    # Chế độ train
    model.train()
    train_fn(train_loader, model, optimizer, loss_fn, scaler, scaled_anchors)

    # TIPS THỰC CHIẾN: Lưu Checkpoint
    # Lưu lại model sau mỗi 10 epoch hoặc khi loss giảm để tránh mất dữ liệu nếu Kaggle/Colab bị ngắt kết nối
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    torch.save(checkpoint, "/kaggle/working/yolo_bd100k.pth")
    print(f"Đã lưu an toàn Checkpoint tại Epoch {epoch+1}")

In [ ]:

# torch.save(model.state_dict(), f"yolov3_bdd100k_without_optimizer.pth")


In [ ]:
# Cell 10: Inference & Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Định nghĩa màu sắc cho 5 loại phương tiện để dễ phân biệt
COLORS = ['red', 'blue', 'green', 'orange', 'purple']
CLASSES = ['car', 'bus', 'truck', 'motorcycle', 'bicycle']

def plot_image(image, boxes):
    """
    Hàm vẽ Bounding Box lên ảnh.
    image: tensor ảnh đã chuyển về numpy array hoặc PIL Image
    boxes: list các box sau khi đã qua NMS [class_pred, prob_score, x, y, w, h]
    """
    # Chuyển tensor ảnh về numpy array và denormalize (nếu cần)
    # Giả sử ảnh đang ở dạng (C, H, W) tensor, ta đưa về (H, W, C)
    if isinstance(image, torch.Tensor):
        # Chuyển từ (C, H, W) sang (H, W, C)
        image = image.permute(1, 2, 0).cpu().numpy()
        
        # DENORMALIZE: Đảo ngược quá trình Normalize của ImageNet
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = std * image + mean
        
        # Kẹp giá trị lại trong khoảng [0, 1] để matplotlib không báo lỗi
        image = np.clip(image, 0, 1)

    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(image)
    # YOLO trả về tọa độ trung tâm (x, y) và chiều rộng/cao (w, h) (Đã chuẩn hóa 0-1)
    # Cần nhân với kích thước ảnh thực tế để vẽ
    height, width, _ = image.shape

    for box in boxes:
        class_pred = int(box[0])
        score = box[1]
        x_center, y_center, w_box, h_box = box[2:]

        print(f"Xe ở tọa độ ({x_center:.2f}, {y_center:.2f}) có W={w_box:.5f}, H={h_box:.5f}")

        # Tính toán tọa độ góc trên cùng bên trái (x_lower_left, y_lower_left)
        upper_left_x = (x_center - w_box / 2) * width
        upper_left_y = (y_center - h_box / 2) * height
        
        box_width = w_box * width
        box_height = h_box * height

        # Tạo khung chữ nhật
        rect = patches.Rectangle(
            (upper_left_x, upper_left_y),
            box_width,
            box_height,
            linewidth=2,
            edgecolor=COLORS[class_pred % len(COLORS)],
            facecolor="none",
        )
        ax.add_patch(rect)
        
        # Thêm Text hiển thị Tên Class và Độ tự tin (Score)
        label = f"{CLASSES[class_pred]}: {score:.2f}"
        plt.text(
            upper_left_x,
            upper_left_y 
            - 5,
            label,
            color='white',
            fontsize=9,
            bbox=dict(facecolor=COLORS[class_pred % len(COLORS)], edgecolor='none', alpha=0.8)
        )

    plt.axis('off')
    plt.show()

def predict_and_plot(image_path, model, anchors, transform, iou_thresh=0.5, prob_thresh=0.5):
    """
    Pipeline dự đoán hoàn chỉnh cho 1 bức ảnh.
    """
    model.eval() # Chuyển model sang chế độ đánh giá (tắt Dropout, BatchNorm tĩnh)
    
    # 1. Đọc ảnh và áp dụng Transform (Validation transform - Không làm méo/lật ảnh)
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # transform này là val_transforms bạn đã định nghĩa ở phần Albumentations
    augmented = transform(image=img, bboxes=[], class_labels=[]) 
    x = augmented["image"].unsqueeze(0).to(device) # Thêm batch dimension: (1, 3, 416, 416)

    # 2. Chạy qua Model
    with torch.no_grad():
        out = model(x)
        
        # Chuẩn bị list để chứa box từ 3 scale
        bboxes = []
        S = [13, 26, 52]
        
        for i in range(3):
            # Ép kiểu list 'anchors[i]' thành PyTorch Tensor và đưa lên GPU
            anchor_tensor = torch.tensor(anchors[i]).to(device)
            # Hàm cells_to_bboxes chuyển output của mạng thành list các box [class, score, x, y, w, h]
            batch_bboxes = cells_to_bboxes(
                out[i], anchor_tensor, S=S[i], is_preds=True
            )
            bboxes += batch_bboxes[0] # Lấy ảnh đầu tiên trong batch (vì batch_size = 1)

    # 3. Lọc Box bằng Non-Max Suppression
    nms_boxes = non_max_suppression(
        bboxes, 
        iou_threshold=iou_thresh, 
        prob_threshold=prob_thresh, 
        box_format="midpoint"
    )

    print(f"Mô hình tìm thấy {len(nms_boxes)} phương tiện.")
    
    # 4. Vẽ ảnh
    plot_image(augmented["image"], nms_boxes)

# ==== THỰC THI ====
# Giả sử bạn đã load model weights (nếu train xong thì model đang nằm trên RAM rồi)
model = YOLOv3(num_classes=5).to(device)

# Tải toàn bộ file checkpoint lên
checkpoint = torch.load("/kaggle/working/yolov3_bdd100k.pth", map_location=device)
# Chỉ nạp phần "state_dict" vào mô hình
model.load_state_dict(checkpoint["state_dict"])

# Chạy thử với 1 ảnh test trong BDD100K
test_img_path = "/kaggle/input/datasets/marquis03/bdd100k/test/cabc30fc-eb673c5a.jpg"
predict_and_plot(test_img_path, model, ANCHORS, val_transforms, prob_thresh=0.8)